In [ ]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')

print("GPU VERIFICATION:")
device = torch.device("cuda" if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 6) else "cpu")
if device.type == 'cuda':
    print(f"SUCCESS: Active GPU detected - {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No compatible GPU detected. Runtime > Change runtime type > T4 GPU.")

repo_dir = "/content/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}

os.chdir(repo_dir)
!chmod +x build_kaggle.sh
!./build_kaggle.sh

In [ ]:
import os

models_dir = "/content/drive/MyDrive/zerocross_models"
os.makedirs(models_dir, exist_ok=True)

existing = os.listdir(models_dir)
print(f"Checking: {models_dir}")
print(f"Files found: {existing if existing else '(empty)'}\n")

if "best_model.pth" in existing or "pipeline_state.json" in existing:
    print("Previous progress detected, pipeline.py will resume automatically.")
else:
    print("No checkpoint here yet. Upload best_model.pth, pipeline_state.json,")
    print(f"training_log.csv, replay_buffer.pt into {models_dir} before training.")

In [ ]:
import json, os, math, torch

models_dir = "/content/drive/MyDrive/zerocross_models"
state_path = os.path.join(models_dir, "pipeline_state.json")
model_path = os.path.join(models_dir, "best_model.pth")

MAX_REJECTIONS = 5

total_iterations = 0
consecutive_rejections = 0
if os.path.exists(state_path):
    state_data = json.load(open(state_path))
    total_iterations = state_data.get("total_iterations", 0)
    consecutive_rejections = state_data.get("consecutive_rejections", 0)

if total_iterations == 0 and os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        total_iterations = ckpt.get("iteration", 0)

next_iter = total_iterations + 1

# Updated 4-Tier Learning Rate Schedule
if next_iter <= 100:
    lr = 0.001
elif next_iter <= 250:
    lr = 0.0005
elif next_iter <= 700:
    lr = 0.0001
else:
    lr = 0.00003

print(f"total_iterations (resume point): {total_iterations}")
print(f"next iteration will be:          {next_iter}")
print(f"learning rate that implies:       {lr}")
print(f"consecutive rejections:           {consecutive_rejections} / {MAX_REJECTIONS}")

stall_boost_at = max(1, math.ceil(MAX_REJECTIONS / 2))
if consecutive_rejections >= MAX_REJECTIONS:
    print("  -> at the forced-promotion threshold: the next rejection only forces")
    print("     a promotion through if win rate vs champion clears --min-force-promote-winrate")
elif consecutive_rejections >= stall_boost_at:
    print(f"  -> past the stall-boost threshold ({stall_boost_at}): evaluations are being")
    print("     widened automatically to cut through noise before any forced promotion")
print()

In [ ]:
!python python/pipeline.py \
    --iterations 100 \
    --concurrent-games 100 \
    --games-per-iteration 400 \
    --mcts-sims 300 \
    --eval-games 100 \
    --eval-sims 200 \
    --batch-size 512 \
    --max-rejections 5 \
    --min-force-promote-winrate 0.52 \
    --stall-eval-multiplier 3 \
    --num-res-blocks 6 \
    --num-channels 128 \
    --max-buffer-size 1000000 \
    --buffer-archive-interval 5

In [ ]:
import json
print(json.load(open("/content/drive/MyDrive/zerocross_models/pipeline_state.json")))